# Customer Shopping Behavior & Purchase Amount Prediction

**IBM SkillsBuild Data Analytics Internship Project**

---

| Property | Detail |
|---|---|
| **Project Title** | Customer Shopping Behavior & Purchase Amount Prediction |
| **Dataset** | shopping_trends.csv (3,900 rows × 19 columns) |
| **Target Variable** | Purchase Amount (USD) |
| **ML Models** | Linear Regression · Random Forest Regressor · XGBoost Regressor |
| **Evaluation Metrics** | MAE · RMSE · R² |
| **Frontend** | Streamlit interactive dashboard |
| **Program** | IBM SkillsBuild Data Analytics Internship |

---

## Project Structure

This notebook contains **six parts**:

1. **Setup & Imports** — install and import all required libraries
2. **Data Loading, Cleaning & Validation** — load `shopping_trends.csv`, inspect and clean
3. **Exploratory Data Analysis (EDA)** — demographics, categories, purchase amounts, payment, discounts, seasonal trends
4. **Machine Learning** — feature engineering, train/test split, train 3 models, evaluate with MAE/RMSE/R²
5. **Model Comparison & Feature Importance** — side-by-side comparison, save best model
6. **Streamlit Dashboard** — complete `app.py` embedded as a code cell ready to run

> **To launch the Streamlit dashboard**, run Part 6's cell which writes `app_combined.py`, then execute:
> ```
> streamlit run app_combined.py
> ```

---
## Part 1 — Setup & Imports

In [ ]:
# Install required libraries (run once if not already installed)
import subprocess, sys

packages = [
    'pandas', 'numpy', 'scikit-learn', 'xgboost',
    'streamlit', 'plotly', 'matplotlib', 'seaborn',
    'joblib', 'statsmodels'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

print('All packages ready.')

In [ ]:
# Core imports
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats as spstats

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

print('Imports complete. Models dir:', MODELS_DIR)

---
## Part 2 — Data Loading, Cleaning & Validation

In [ ]:
# ── 2.1 Load dataset ─────────────────────────────────────────────────────────
DATA_PATH = 'shopping_trends.csv'
df_raw = pd.read_csv(DATA_PATH)

print('=' * 60)
print('  Customer Shopping Behavior — Dataset Overview')
print('=' * 60)
print(f'Shape        : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Columns      : {list(df_raw.columns)}')
print(f'Missing vals : {df_raw.isnull().sum().sum()}')
print(f'Duplicates   : {df_raw.duplicated().sum()}')
print()
df_raw.head()

In [ ]:
# ── 2.2 Data types and statistics ────────────────────────────────────────────
print('Data Types:')
print(df_raw.dtypes)
print()
print('Numerical Statistics:')
df_raw.describe().T.round(2)

In [ ]:
# ── 2.3 Missing value analysis ───────────────────────────────────────────────
missing_df = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing Count': df_raw.isnull().sum().values,
    'Missing %': (df_raw.isnull().sum().values / len(df_raw) * 100).round(2)
})
print('Missing Values per Column:')
print(missing_df.to_string(index=False))

if missing_df['Missing Count'].sum() == 0:
    print('\n✅ No missing values found in the dataset.')
else:
    print(f'\n⚠️ Total missing: {missing_df["Missing Count"].sum()}')

In [ ]:
# ── 2.4 Duplicate check ──────────────────────────────────────────────────────
dupes = df_raw.duplicated().sum()
if dupes == 0:
    print('✅ No duplicate rows found.')
else:
    print(f'⚠️ {dupes} duplicate row(s) found.')

In [ ]:
# ── 2.5 Clean data ───────────────────────────────────────────────────────────
df = df_raw.copy()

# Remove duplicates
df.drop_duplicates(inplace=True)

# Drop Customer ID (non-predictive identifier)
if 'Customer ID' in df.columns:
    df.drop(columns=['Customer ID'], inplace=True)

# Drop any remaining nulls
df.dropna(inplace=True)

print(f'After cleaning : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns kept   : {list(df.columns)}')

# Save cleaned data
df.to_csv(os.path.join(MODELS_DIR, 'cleaned_data.csv'), index=False)
print(f'Cleaned dataset saved → {MODELS_DIR}/cleaned_data.csv')

In [ ]:
# ── 2.6 Target variable summary ──────────────────────────────────────────────
TARGET = 'Purchase Amount (USD)'

print('Target Variable Summary — Purchase Amount (USD):')
print(df[TARGET].describe().round(2))
print(f'\nCorrelation with Age              : {df["Age"].corr(df[TARGET]):.4f}')
print(f'Correlation with Review Rating    : {df["Review Rating"].corr(df[TARGET]):.4f}')
print(f'Correlation with Previous Purchases: {df["Previous Purchases"].corr(df[TARGET]):.4f}')

---
## Part 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Customer Demographics — Age distribution ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Demographics', fontsize=14, fontweight='bold')

axes[0].hist(df['Age'], bins=20, color='#2E75B6', edgecolor='white')
axes[0].axvline(df['Age'].mean(), color='#ED7D31', linestyle='--', linewidth=2,
                label=f'Mean: {df["Age"].mean():.1f}')
axes[0].axvline(df['Age'].median(), color='#70AD47', linestyle='--', linewidth=2,
                label=f'Median: {int(df["Age"].median())}')
axes[0].set_title('Age Distribution of Customers')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')
axes[0].legend()

gc = df['Gender'].value_counts()
axes[1].pie(gc.values, labels=gc.index, autopct='%1.1f%%',
            colors=['#2E75B6', '#ED7D31'],
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Customer Gender Distribution')

plt.tight_layout()
plt.show()
print(f'Age range: {int(df["Age"].min())}–{int(df["Age"].max())}  |  Mean: {df["Age"].mean():.2f}  |  StdDev: {df["Age"].std():.2f}')
print(f'Gender counts: {dict(gc)}')

In [ ]:
# ── 3.2 Top locations & avg spend by gender ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_loc = df['Location'].value_counts().head(10)
axes[0].barh(list(top_loc.index), top_loc.values, color='#2E75B6')
axes[0].set_title('Top 10 Customer Locations')
axes[0].set_xlabel('Number of Customers')
axes[0].invert_yaxis()
for i, v in enumerate(top_loc.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

gs = df.groupby('Gender')[TARGET].mean().round(2)
bars = axes[1].bar(gs.index, gs.values, color=['#2E75B6', '#ED7D31'],
                   edgecolor='white', width=0.4)
axes[1].set_title('Average Purchase Amount by Gender')
axes[1].set_ylabel('Average Amount (USD)')
axes[1].set_ylim(55, 65)
for bar, val in zip(bars, gs.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                 f'${val:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Product & Category Analysis ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Product & Category Analysis', fontsize=14, fontweight='bold')

cat_counts = df['Category'].value_counts()
axes[0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=['#2E75B6', '#ED7D31', '#70AD47', '#7030A0'],
            wedgeprops={'edgecolor': 'white', 'linewidth': 2}, startangle=140)
axes[0].set_title('Purchase Volume by Category')

cat_spend = df.groupby('Category')[TARGET].mean().sort_values(ascending=False).round(2)
bars = axes[1].bar(cat_spend.index, cat_spend.values,
                   color=['#2E75B6', '#70AD47', '#ED7D31', '#7030A0'],
                   edgecolor='white')
axes[1].set_title('Average Purchase Amount by Category (USD)')
axes[1].set_ylabel('Average Amount (USD)')
axes[1].set_ylim(50, 65)
for bar, val in zip(bars, cat_spend.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                 f'${val:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print('Category counts:', dict(cat_counts))
print('Category avg spend:', dict(cat_spend))

In [ ]:
# ── 3.4 Top purchased items ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_items = df['Item Purchased'].value_counts().head(10)
axes[0].barh(list(top_items.index), top_items.values, color='#2E75B6')
axes[0].set_title('Top 10 Most Purchased Items')
axes[0].set_xlabel('Purchase Count')
axes[0].invert_yaxis()
for i, v in enumerate(top_items.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

item_spend = (df.groupby('Item Purchased')[TARGET].mean()
              .sort_values(ascending=False).head(10).round(2))
axes[1].barh(list(item_spend.index), item_spend.values, color='#00B0F0')
axes[1].set_title('Top 10 Items by Average Purchase Amount')
axes[1].set_xlabel('Average Amount (USD)')
axes[1].invert_yaxis()
for i, v in enumerate(item_spend.values):
    axes[1].text(v + 0.1, i, f'${v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Purchase Amount Distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Purchase Amount Analysis', fontsize=14, fontweight='bold')

axes[0].hist(df[TARGET], bins=30, color='#70AD47', edgecolor='white')
axes[0].axvline(df[TARGET].mean(), color='#ED7D31', linestyle='--', linewidth=2,
                label=f'Mean: ${df[TARGET].mean():.2f}')
axes[0].axvline(df[TARGET].median(), color='#C00000', linestyle='--', linewidth=2,
                label=f'Median: ${int(df[TARGET].median())}')
axes[0].set_title('Distribution of Purchase Amount (USD)')
axes[0].set_xlabel('Purchase Amount (USD)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

categories = list(df['Category'].unique())
data_by_cat = [df[df['Category'] == c][TARGET].values for c in categories]
bp = axes[1].boxplot(data_by_cat, patch_artist=True,
                     tick_labels=categories, notch=False)
colors = ['#2E75B6', '#ED7D31', '#70AD47', '#7030A0']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Purchase Amount by Category')
axes[1].set_ylabel('Purchase Amount (USD)')

plt.tight_layout()
plt.show()
print(df[TARGET].describe().round(2))

In [ ]:
# ── 3.6 Review Rating & Previous Purchases vs Purchase Amount (scatter) ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Scatter Plots — Numerical Features vs Purchase Amount', fontsize=14, fontweight='bold')

sample = df.sample(min(1500, len(df)), random_state=42)

axes[0].scatter(sample['Review Rating'], sample[TARGET],
                alpha=0.3, color='#7030A0', s=20)
slope1, intercept1, r1, p1, _ = spstats.linregress(df['Review Rating'], df[TARGET])
x1 = np.linspace(df['Review Rating'].min(), df['Review Rating'].max(), 100)
axes[0].plot(x1, slope1 * x1 + intercept1, color='#C00000', linewidth=2,
             label=f'OLS  r={r1:.4f}')
axes[0].set_title(f'Review Rating vs Purchase Amount  (r={r1:.4f})')
axes[0].set_xlabel('Review Rating (1.0–5.0)')
axes[0].set_ylabel('Purchase Amount (USD)')
axes[0].legend()

axes[1].scatter(sample['Previous Purchases'], sample[TARGET],
                alpha=0.3, color='#00B0F0', s=20)
slope2, intercept2, r2_corr, p2, _ = spstats.linregress(df['Previous Purchases'], df[TARGET])
x2 = np.linspace(df['Previous Purchases'].min(), df['Previous Purchases'].max(), 100)
axes[1].plot(x2, slope2 * x2 + intercept2, color='#C00000', linewidth=2,
             label=f'OLS  r={r2_corr:.4f}')
axes[1].set_title(f'Previous Purchases vs Purchase Amount  (r={r2_corr:.4f})')
axes[1].set_xlabel('Number of Previous Purchases')
axes[1].set_ylabel('Purchase Amount (USD)')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Both correlations are near-zero — consistent with a synthetically generated uniform target.')

In [ ]:
# ── 3.7 Payment Method & Shipping Type ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Payment Method & Shipping Type Analysis', fontsize=14, fontweight='bold')
COLORS6 = ['#2E75B6', '#ED7D31', '#70AD47', '#7030A0', '#00B0F0', '#FFC000']

pm = df['Payment Method'].value_counts()
axes[0, 0].pie(pm.values, labels=pm.index, autopct='%1.1f%%',
               colors=COLORS6, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0, 0].set_title('Payment Method Distribution')

pm_spend = df.groupby('Payment Method')[TARGET].mean().sort_values(ascending=False).round(2)
axes[0, 1].barh(list(pm_spend.index), pm_spend.values, color='#2E75B6')
axes[0, 1].set_title('Avg Purchase Amount by Payment Method')
axes[0, 1].set_xlabel('Average (USD)')
axes[0, 1].set_xlim(55, 65)
axes[0, 1].invert_yaxis()
for i, v in enumerate(pm_spend.values):
    axes[0, 1].text(v + 0.05, i, f'${v:.2f}', va='center', fontsize=9)

sh = df['Shipping Type'].value_counts()
axes[1, 0].bar(sh.index, sh.values, color=COLORS6[:len(sh)], edgecolor='white')
axes[1, 0].set_title('Shipping Type Distribution')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=20)
for i, (idx, val) in enumerate(zip(sh.index, sh.values)):
    axes[1, 0].text(i, val + 2, str(val), ha='center', fontsize=9)

sh_spend = df.groupby('Shipping Type')[TARGET].mean().sort_values(ascending=False).round(2)
axes[1, 1].bar(sh_spend.index, sh_spend.values,
               color=COLORS6[:len(sh_spend)], edgecolor='white')
axes[1, 1].set_title('Avg Purchase Amount by Shipping Type')
axes[1, 1].set_ylabel('Average (USD)')
axes[1, 1].set_ylim(55, 65)
axes[1, 1].tick_params(axis='x', rotation=20)
for i, (idx, val) in enumerate(zip(sh_spend.index, sh_spend.values)):
    axes[1, 1].text(i, val + 0.05, f'${val:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.8 Subscription & Discount Analysis ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Subscription & Discount Analysis', fontsize=14, fontweight='bold')

sub = df['Subscription Status'].value_counts()
axes[0, 0].pie(sub.values, labels=sub.index, autopct='%1.1f%%',
               colors=['#70AD47', '#C00000'],
               wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0, 0].set_title('Subscription Status Distribution')

sub_spend = df.groupby('Subscription Status')[TARGET].mean().round(2)
bars = axes[0, 1].bar(sub_spend.index, sub_spend.values,
                      color=['#70AD47', '#C00000'], edgecolor='white', width=0.4)
axes[0, 1].set_title('Avg Spend: Subscriber vs Non-Subscriber')
axes[0, 1].set_ylabel('Average Amount (USD)')
axes[0, 1].set_ylim(55, 65)
for bar, val in zip(bars, sub_spend.values):
    axes[0, 1].text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                    f'${val:.2f}', ha='center', fontsize=12, fontweight='bold')

disc_spend = df.groupby('Discount Applied')[TARGET].mean().round(2)
bars2 = axes[1, 0].bar(disc_spend.index, disc_spend.values,
                       color=['#2E75B6', '#ED7D31'], edgecolor='white', width=0.4)
axes[1, 0].set_title('Avg Spend: Discount Applied vs Not')
axes[1, 0].set_ylabel('Average Amount (USD)')
axes[1, 0].set_ylim(55, 65)
for bar, val in zip(bars2, disc_spend.values):
    axes[1, 0].text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                    f'${val:.2f}', ha='center', fontsize=12, fontweight='bold')

promo_spend = df.groupby('Promo Code Used')[TARGET].mean().round(2)
bars3 = axes[1, 1].bar(promo_spend.index, promo_spend.values,
                       color=['#7030A0', '#00B0F0'], edgecolor='white', width=0.4)
axes[1, 1].set_title('Avg Spend: Promo Code Used vs Not')
axes[1, 1].set_ylabel('Average Amount (USD)')
axes[1, 1].set_ylim(55, 65)
for bar, val in zip(bars3, promo_spend.values):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                    f'${val:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('Subscription impact:', dict(sub_spend))
print('Discount impact    :', dict(disc_spend))
print('Promo code impact  :', dict(promo_spend))

In [ ]:
# ── 3.9 Seasonal Trends ───────────────────────────────────────────────────────
SEASON_ORDER = ['Spring', 'Summer', 'Fall', 'Winter']
SEASON_COLS  = ['#70AD47', '#00B0F0', '#ED7D31', '#2E75B6']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Seasonal Shopping Trends', fontsize=14, fontweight='bold')

sc = df['Season'].value_counts().reindex(SEASON_ORDER)
axes[0, 0].bar(sc.index, sc.values, color=SEASON_COLS, edgecolor='white')
axes[0, 0].set_title('Purchase Count by Season')
axes[0, 0].set_ylabel('Number of Purchases')
for i, v in enumerate(sc.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center', fontsize=10, fontweight='bold')

ss = df.groupby('Season')[TARGET].mean().reindex(SEASON_ORDER).round(2)
bars = axes[0, 1].bar(ss.index, ss.values, color=SEASON_COLS, edgecolor='white')
axes[0, 1].set_title('Average Purchase Amount by Season (USD)')
axes[0, 1].set_ylabel('Average Amount (USD)')
axes[0, 1].set_ylim(55, 65)
for bar, val in zip(bars, ss.values):
    axes[0, 1].text(bar.get_x() + bar.get_width() / 2, val + 0.1,
                    f'${val:.2f}', ha='center', fontsize=10, fontweight='bold')

freq_order = ['Weekly', 'Fortnightly', 'Bi-Weekly', 'Monthly',
              'Quarterly', 'Every 3 Months', 'Annually']
freq = df['Frequency of Purchases'].value_counts().reindex(freq_order)
axes[1, 0].barh(list(freq.index), freq.values, color='#2E75B6')
axes[1, 0].set_title('Purchase Frequency Distribution')
axes[1, 0].set_xlabel('Count')
axes[1, 0].invert_yaxis()
for i, v in enumerate(freq.values):
    axes[1, 0].text(v + 1, i, str(v), va='center', fontsize=9)

cat_season = df.groupby(['Season', 'Category'])[TARGET].count().unstack()
cat_season.reindex(SEASON_ORDER).plot(
    kind='bar', ax=axes[1, 1],
    color=['#2E75B6', '#ED7D31', '#70AD47', '#7030A0'],
    edgecolor='white', width=0.7
)
axes[1, 1].set_title('Category Purchase Volume by Season')
axes[1, 1].set_xlabel('Season')
axes[1, 1].set_ylabel('Purchase Count')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].legend(title='Category', fontsize=8)

plt.tight_layout()
plt.show()
print('Avg spend by season:', dict(ss))

---
## Part 4 — Machine Learning: Feature Engineering & Model Training

In [ ]:
# ── 4.1 Define features & target ─────────────────────────────────────────────
TARGET       = 'Purchase Amount (USD)'
FEATURE_COLS = [c for c in df.columns if c != TARGET]

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Target column      : {TARGET}')
print(f'Total features     : {len(FEATURE_COLS)}')
print(f'Categorical ({len(categorical_cols)}): {categorical_cols}')
print(f'Numerical   ({len(numerical_cols)}): {numerical_cols}')

In [ ]:
# ── 4.2 Label encode categorical columns ─────────────────────────────────────
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

joblib.dump(label_encoders, os.path.join(MODELS_DIR, 'label_encoders.pkl'))
print(f'LabelEncoder applied to {len(categorical_cols)} columns')
print(f'Saved → {MODELS_DIR}/label_encoders.pkl')

In [ ]:
# ── 4.3 Scale numerical columns ──────────────────────────────────────────────
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

joblib.dump(scaler,           os.path.join(MODELS_DIR, 'scaler.pkl'))
joblib.dump(FEATURE_COLS,     os.path.join(MODELS_DIR, 'feature_names.pkl'))
joblib.dump(categorical_cols, os.path.join(MODELS_DIR, 'categorical_cols.pkl'))
joblib.dump(numerical_cols,   os.path.join(MODELS_DIR, 'numerical_cols.pkl'))
print('StandardScaler applied to numerical columns')
print(f'Saved → {MODELS_DIR}/scaler.pkl')

In [ ]:
# ── 4.4 Train / test split ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size : {X_train.shape[0]:,} samples')
print(f'Test  size : {X_test.shape[0]:,} samples')
print(f'Features   : {X_train.shape[1]}')

In [ ]:
# ── 4.5 Train all three models ────────────────────────────────────────────────
models_def = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
}

results      = []
trained_models = {}

print('Training models...')
print('-' * 55)

for name, model in models_def.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MAE': round(mae, 4),
                    'RMSE': round(rmse, 4), 'R2': round(r2, 4)})
    trained_models[name] = model

    print(f'  {name:<22}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

results_df = pd.DataFrame(results)
print('-' * 55)
print('Done.')

---
## Part 5 — Model Comparison, Evaluation & Feature Importance

In [ ]:
# ── 5.1 Results table ─────────────────────────────────────────────────────────
print('\nModel Evaluation Results (Test Set — 780 samples):')
print('=' * 55)
print(results_df.to_string(index=False))
print('=' * 55)

best_row  = results_df.loc[results_df['RMSE'].idxmin()]
best_name = best_row['Model']
print(f'\n✅ Best model: {best_name}  (RMSE={best_row["RMSE"]})')
results_df

In [ ]:
# ── 5.2 Model comparison chart ────────────────────────────────────────────────
models_list = results_df['Model'].tolist()
mae_vals    = results_df['MAE'].tolist()
rmse_vals   = results_df['RMSE'].tolist()
r2_vals     = results_df['R2'].tolist()
x           = np.arange(len(models_list))
bar_colors  = ['#2E75B6', '#70AD47', '#ED7D31']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('ML Model Comparison — Actual Test-Set Results', fontsize=14, fontweight='bold')

axes[0].bar(x, mae_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[0].set_title('MAE (lower is better)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_list, rotation=10, fontsize=9)
axes[0].set_ylabel('MAE (USD)')
for i, v in enumerate(mae_vals):
    axes[0].text(i, v + 0.05, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
best_mae_idx = mae_vals.index(min(mae_vals))
axes[0].patches[best_mae_idx].set_edgecolor('#FFC000')
axes[0].patches[best_mae_idx].set_linewidth(3)

axes[1].bar(x, rmse_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[1].set_title('RMSE (lower is better)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_list, rotation=10, fontsize=9)
axes[1].set_ylabel('RMSE (USD)')
for i, v in enumerate(rmse_vals):
    axes[1].text(i, v + 0.05, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
best_rmse_idx = rmse_vals.index(min(rmse_vals))
axes[1].patches[best_rmse_idx].set_edgecolor('#FFC000')
axes[1].patches[best_rmse_idx].set_linewidth(3)

axes[2].bar(x, r2_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[2].set_title('R2 Score (higher is better)')
axes[2].set_xticks(x)
axes[2].set_xticklabels(models_list, rotation=10, fontsize=9)
axes[2].set_ylabel('R2 Score')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
for i, v in enumerate(r2_vals):
    offset = 0.003 if v >= 0 else -0.012
    axes[2].text(i, v + offset, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

fig.text(0.5, -0.02,
         'Gold border = best value  |  Best model: ' + best_name + ' (lowest RMSE)',
         ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Save all model artifacts ─────────────────────────────────────────────
best_model = trained_models[best_name]

joblib.dump(best_model, os.path.join(MODELS_DIR, 'best_model.pkl'))
joblib.dump(best_name,  os.path.join(MODELS_DIR, 'best_model_name.pkl'))

for name, model in trained_models.items():
    safe = name.lower().replace(' ', '_')
    joblib.dump(model, os.path.join(MODELS_DIR, f'{safe}.pkl'))

results_df.to_csv(os.path.join(MODELS_DIR, 'model_results.csv'), index=False)

print('All model artifacts saved to models/:')
for f in sorted(os.listdir(MODELS_DIR)):
    size = os.path.getsize(os.path.join(MODELS_DIR, f))
    print(f'  {f:<35} {size:>10,} bytes')

In [ ]:
# ── 5.4 Feature Importance ────────────────────────────────────────────────────
rf_model  = trained_models['Random Forest']
xgb_model = trained_models['XGBoost']

rf_imp_df = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(12)

xgb_imp_df = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Analysis (Top 12 Features)', fontsize=14, fontweight='bold')

rf_feats  = list(rf_imp_df['Feature'])[::-1]
rf_scores = list(rf_imp_df['Importance'])[::-1]
axes[0].barh(rf_feats, rf_scores, color='#2E75B6')
axes[0].set_title('Random Forest — Feature Importance')
axes[0].set_xlabel('Importance Score')
for i, v in enumerate(rf_scores):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

xgb_feats  = list(xgb_imp_df['Feature'])[::-1]
xgb_scores = list(xgb_imp_df['Importance'])[::-1]
axes[1].barh(xgb_feats, xgb_scores, color='#ED7D31')
axes[1].set_title('XGBoost — Feature Importance')
axes[1].set_xlabel('Importance Score')
for i, v in enumerate(xgb_scores):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print('\nRandom Forest — Top 8:')
print(rf_imp_df.head(8).to_string(index=False))
print('\nXGBoost — Top 8:')
print(xgb_imp_df.head(8).to_string(index=False))

---
## Part 6 — Streamlit Dashboard

The cell below writes the complete `app_combined.py` file.

**After running this cell**, launch the dashboard from a terminal:
```
streamlit run app_combined.py
```

The dashboard includes 12 sections:
- Project Overview
- Dataset Summary
- Data Cleaning & Validation
- Customer Demographics
- Product & Category Analysis
- Purchase Amount Analysis
- Payment & Shipping Analysis
- Subscription & Discounts
- Seasonal Trends
- ML Model Comparison
- Feature Importance
- Live Prediction Form

In [ ]:
# ── 6.1 Write app_combined.py ─────────────────────────────────────────────────
# This cell writes the complete Streamlit app to disk.
# Then run:  streamlit run app_combined.py

app_code = '''
"""
app_combined.py
===============
Customer Shopping Behavior & Purchase Amount Prediction
IBM SkillsBuild Data Analytics Internship Project

Run:  streamlit run app_combined.py
"""

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="Shopping Trends Prediction",
    page_icon="\\U0001f6cd",
    layout="wide",
    initial_sidebar_state="expanded",
)

DATA_PATH  = "shopping_trends.csv"
MODELS_DIR = "models"
TARGET     = "Purchase Amount (USD)"


# ── Training pipeline (runs once, caches artifacts) ──────────────────────────
@st.cache_resource(show_spinner="Training models — please wait...")
def run_training_pipeline():
    import numpy as _np
    os.makedirs(MODELS_DIR, exist_ok=True)
    df = pd.read_csv(DATA_PATH)
    df.drop_duplicates(inplace=True)
    if "Customer ID" in df.columns:
        df.drop(columns=["Customer ID"], inplace=True)
    df.dropna(inplace=True)
    df.to_csv(os.path.join(MODELS_DIR, "cleaned_data.csv"), index=False)

    feat_cols = [c for c in df.columns if c != TARGET]
    X = df[feat_cols].copy()
    y = df[TARGET].copy()

    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

    scaler = StandardScaler()
    X[num_cols] = scaler.fit_transform(X[num_cols])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest":     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost":           XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    }

    results = []
    trained = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mae  = mean_absolute_error(y_test, y_pred)
        rmse = _np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)
        results.append({"Model": name, "MAE": round(mae, 4),
                         "RMSE": round(rmse, 4), "R2": round(r2, 4)})
        trained[name] = model
        safe = name.lower().replace(" ", "_")
        joblib.dump(model, os.path.join(MODELS_DIR, f"{safe}.pkl"))

    res_df = pd.DataFrame(results)
    res_df.to_csv(os.path.join(MODELS_DIR, "model_results.csv"), index=False)

    best_name  = res_df.loc[res_df["RMSE"].idxmin(), "Model"]
    best_model = trained[best_name]
    joblib.dump(best_model,    os.path.join(MODELS_DIR, "best_model.pkl"))
    joblib.dump(best_name,     os.path.join(MODELS_DIR, "best_model_name.pkl"))
    joblib.dump(label_encoders, os.path.join(MODELS_DIR, "label_encoders.pkl"))
    joblib.dump(scaler,         os.path.join(MODELS_DIR, "scaler.pkl"))
    joblib.dump(feat_cols,      os.path.join(MODELS_DIR, "feature_names.pkl"))
    joblib.dump(cat_cols,       os.path.join(MODELS_DIR, "categorical_cols.pkl"))
    joblib.dump(num_cols,       os.path.join(MODELS_DIR, "numerical_cols.pkl"))

    return {
        "best_model":        best_model,
        "best_model_name":   best_name,
        "label_encoders":    label_encoders,
        "scaler":            scaler,
        "feature_names":     feat_cols,
        "categorical_cols":  cat_cols,
        "numerical_cols":    num_cols,
        "results_df":        res_df,
        "rf_model":          trained["Random Forest"],
        "xgb_model":         trained["XGBoost"],
    }


@st.cache_data(show_spinner=False)
def load_raw_data():
    return pd.read_csv(DATA_PATH)


# ── Run pipeline ─────────────────────────────────────────────────────────────
artifacts = run_training_pipeline()
raw_df    = load_raw_data()
eda_df    = raw_df.drop(columns=["Customer ID"] if "Customer ID" in raw_df.columns else [],
                         errors="ignore")
results_df = artifacts["results_df"]

# ── Sidebar navigation ────────────────────────────────────────────────────────
SECTIONS = [
    "\\U0001f3e0 Project Overview",
    "\\U0001f4cb Dataset Summary",
    "\\U0001f9f9 Data Cleaning & Validation",
    "\\U0001f465 Customer Demographics",
    "\\U0001f6d2 Product & Category Analysis",
    "\\U0001f4b0 Purchase Amount Analysis",
    "\\U0001f4b3 Payment & Shipping Analysis",
    "\\U0001f381 Subscription & Discounts",
    "\\U0001f4c5 Seasonal Trends",
    "\\U0001f916 ML Model Comparison",
    "\\U0001f4ca Feature Importance",
    "\\U0001f52e Live Prediction",
]

st.sidebar.image("https://upload.wikimedia.org/wikipedia/commons/5/51/IBM_logo.svg", width=120)
st.sidebar.title("Shopping Trends")
st.sidebar.caption("IBM SkillsBuild Internship Project")
st.sidebar.markdown("---")
selected = st.sidebar.radio("Navigate", SECTIONS, label_visibility="collapsed")
st.sidebar.markdown("---")
st.sidebar.caption("Dataset: shopping_trends.csv")
st.sidebar.caption("Target: Purchase Amount (USD)")
st.sidebar.caption(f"Best Model: {artifacts[\'best_model_name\']}")


# ============================================================
# SECTIONS
# ============================================================
if "Project Overview" in selected:
    st.title("Customer Shopping Behavior & Purchase Amount Prediction")
    st.markdown("### IBM SkillsBuild Data Analytics Internship Project")
    st.markdown("---")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total Records", f"{len(raw_df):,}")
    c2.metric("Features", str(raw_df.shape[1] - 2))
    c3.metric("Target", "Purchase Amount")
    c4.metric("ML Models", "3")
    st.markdown("---")
    st.markdown("""
    ## Project Objective
    Analyze customer shopping behavior and predict **Purchase Amount (USD)** using machine learning.

    ## Technology Stack
    Python | Pandas | NumPy | Scikit-learn | XGBoost | Streamlit | Plotly

    ## Models Built
    Linear Regression | Random Forest Regressor | XGBoost Regressor

    ## Metrics
    MAE | RMSE | R2
    """)
    best = artifacts[\'best_model_name\']
    best_row = results_df[results_df["Model"] == best].iloc[0]
    st.success(f"Best model: **{best}**  |  MAE={best_row[\'MAE\']}  RMSE={best_row[\'RMSE\']}  R2={best_row[\'R2\']}")


elif "Dataset Summary" in selected:
    st.title("Dataset Summary")
    st.markdown("---")
    c1, c2 = st.columns(2)
    c1.metric("Rows", f"{raw_df.shape[0]:,}")
    c2.metric("Columns", str(raw_df.shape[1]))
    st.markdown("### Sample Rows")
    st.dataframe(raw_df.head(10), use_container_width=True)
    st.markdown("### Column Information")
    info_df = pd.DataFrame({
        "Column":    raw_df.columns,
        "Data Type": raw_df.dtypes.astype(str).values,
        "Non-Null":  raw_df.notnull().sum().values,
        "Null":      raw_df.isnull().sum().values,
        "Unique":    raw_df.nunique().values,
    })
    st.dataframe(info_df, use_container_width=True)
    st.markdown("### Descriptive Statistics")
    st.dataframe(raw_df.describe().T.round(2), use_container_width=True)


elif "Data Cleaning" in selected:
    st.title("Data Cleaning & Validation")
    st.markdown("---")
    c1, c2, c3 = st.columns(3)
    c1.metric("Total Rows", f"{len(raw_df):,}")
    c2.metric("Duplicates", str(raw_df.duplicated().sum()))
    c3.metric("Missing Values", str(raw_df.isnull().sum().sum()))
    if raw_df.isnull().sum().sum() == 0:
        st.success("No missing values found.")
    if raw_df.duplicated().sum() == 0:
        st.success("No duplicate rows found.")
    st.markdown("### Cleaning Steps")
    st.markdown("""
    | Step | Action |
    |---|---|
    | Remove Duplicates | `df.drop_duplicates()` |
    | Drop Customer ID | Non-predictive identifier removed |
    | Drop Nulls | Any remaining null rows removed |
    | Label Encoding | 14 categorical columns encoded with LabelEncoder |
    | Standard Scaling | 3 numerical columns scaled with StandardScaler |
    """)
    cat_cols = raw_df.select_dtypes(include="object").columns.tolist()
    chosen = st.selectbox("Inspect a column", cat_cols)
    vc = raw_df[chosen].value_counts().reset_index()
    vc.columns = [chosen, "Count"]
    st.plotly_chart(px.bar(vc, x=chosen, y="Count", color="Count",
                           title=f"Value Counts: {chosen}",
                           color_continuous_scale="Blues"), use_container_width=True)


elif "Demographics" in selected:
    st.title("Customer Demographics")
    st.markdown("---")
    st.plotly_chart(px.histogram(eda_df, x="Age", nbins=20,
                                  title="Age Distribution",
                                  color_discrete_sequence=["#3b82f6"]),
                    use_container_width=True)
    c1, c2 = st.columns(2)
    gc = eda_df["Gender"].value_counts().reset_index()
    gc.columns = ["Gender", "Count"]
    c1.plotly_chart(px.pie(gc, names="Gender", values="Count",
                            title="Gender Distribution", hole=0.4,
                            color_discrete_sequence=["#3b82f6", "#f59e0b"]),
                    use_container_width=True)
    gs = eda_df.groupby("Gender")[TARGET].mean().reset_index()
    gs.columns = ["Gender", "Avg (USD)"]
    gs["Avg (USD)"] = gs["Avg (USD)"].round(2)
    c2.plotly_chart(px.bar(gs, x="Gender", y="Avg (USD)",
                            title="Avg Spend by Gender", color="Gender",
                            text="Avg (USD)",
                            color_discrete_sequence=["#3b82f6", "#f59e0b"]),
                    use_container_width=True)
    lc = eda_df["Location"].value_counts().head(10).reset_index()
    lc.columns = ["Location", "Count"]
    st.plotly_chart(px.bar(lc, x="Count", y="Location", orientation="h",
                            title="Top 10 Locations", color="Count",
                            color_continuous_scale="Blues"), use_container_width=True)


elif "Category" in selected:
    st.title("Product & Category Analysis")
    st.markdown("---")
    c1, c2 = st.columns(2)
    cc = eda_df["Category"].value_counts().reset_index()
    cc.columns = ["Category", "Count"]
    c1.plotly_chart(px.pie(cc, names="Category", values="Count",
                            title="Category Distribution", hole=0.35),
                    use_container_width=True)
    cs = eda_df.groupby("Category")[TARGET].mean().reset_index()
    cs.columns = ["Category", "Avg (USD)"]
    cs["Avg (USD)"] = cs["Avg (USD)"].round(2)
    c2.plotly_chart(px.bar(cs, x="Category", y="Avg (USD)",
                            title="Avg Spend by Category",
                            color="Avg (USD)", color_continuous_scale="Viridis",
                            text="Avg (USD)"), use_container_width=True)
    ti = eda_df["Item Purchased"].value_counts().head(10).reset_index()
    ti.columns = ["Item", "Count"]
    st.plotly_chart(px.bar(ti, x="Count", y="Item", orientation="h",
                            title="Top 10 Items", color="Count",
                            color_continuous_scale="Blues"), use_container_width=True)
    pivot = eda_df.pivot_table(index="Category", columns="Size",
                                values=TARGET, aggfunc="count")
    st.plotly_chart(px.imshow(pivot, title="Category x Size Heatmap",
                               color_continuous_scale="Blues", text_auto=True),
                    use_container_width=True)


elif "Purchase Amount" in selected:
    st.title("Purchase Amount Analysis")
    st.markdown("---")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Min",    f"${eda_df[TARGET].min():.2f}")
    c2.metric("Max",    f"${eda_df[TARGET].max():.2f}")
    c3.metric("Mean",   f"${eda_df[TARGET].mean():.2f}")
    c4.metric("Median", f"${eda_df[TARGET].median():.2f}")
    st.plotly_chart(px.histogram(eda_df, x=TARGET, nbins=30,
                                  title="Purchase Amount Distribution",
                                  color_discrete_sequence=["#10b981"]),
                    use_container_width=True)
    c1b, c2b = st.columns(2)
    c1b.plotly_chart(px.box(eda_df, x="Category", y=TARGET,
                             title="By Category", color="Category"),
                     use_container_width=True)
    c2b.plotly_chart(px.box(eda_df, x="Gender", y=TARGET,
                             title="By Gender", color="Gender",
                             color_discrete_sequence=["#3b82f6", "#f59e0b"]),
                     use_container_width=True)
    st.plotly_chart(px.violin(eda_df, x="Size", y=TARGET,
                               title="By Size", color="Size", box=True),
                    use_container_width=True)
    st.plotly_chart(px.scatter(eda_df, x="Review Rating", y=TARGET,
                                title="Review Rating vs Purchase Amount",
                                opacity=0.5, trendline="ols",
                                color_discrete_sequence=["#6366f1"]),
                    use_container_width=True)


elif "Payment" in selected:
    st.title("Payment & Shipping Analysis")
    st.markdown("---")
    c1, c2 = st.columns(2)
    pm = eda_df["Payment Method"].value_counts().reset_index()
    pm.columns = ["Payment Method", "Count"]
    c1.plotly_chart(px.pie(pm, names="Payment Method", values="Count",
                            title="Payment Method Distribution", hole=0.35),
                    use_container_width=True)
    pms = eda_df.groupby("Payment Method")[TARGET].mean().reset_index()
    pms.columns = ["Payment Method", "Avg (USD)"]
    pms["Avg (USD)"] = pms["Avg (USD)"].round(2)
    c2.plotly_chart(px.bar(pms.sort_values("Avg (USD)", ascending=False),
                            x="Avg (USD)", y="Payment Method", orientation="h",
                            title="Avg Spend by Payment Method",
                            color="Avg (USD)", color_continuous_scale="Teal",
                            text="Avg (USD)"), use_container_width=True)
    c3, c4 = st.columns(2)
    sc2 = eda_df["Shipping Type"].value_counts().reset_index()
    sc2.columns = ["Shipping Type", "Count"]
    c3.plotly_chart(px.bar(sc2, x="Shipping Type", y="Count",
                            title="Shipping Type Counts",
                            color="Count", color_continuous_scale="Blues"),
                    use_container_width=True)
    shs = eda_df.groupby("Shipping Type")[TARGET].mean().reset_index()
    shs.columns = ["Shipping Type", "Avg (USD)"]
    shs["Avg (USD)"] = shs["Avg (USD)"].round(2)
    c4.plotly_chart(px.bar(shs.sort_values("Avg (USD)", ascending=False),
                            x="Shipping Type", y="Avg (USD)",
                            title="Avg Spend by Shipping Type",
                            color="Avg (USD)", color_continuous_scale="Viridis",
                            text="Avg (USD)"), use_container_width=True)


elif "Subscription" in selected:
    st.title("Subscription & Discounts")
    st.markdown("---")
    c1, c2 = st.columns(2)
    sub = eda_df["Subscription Status"].value_counts().reset_index()
    sub.columns = ["Subscription Status", "Count"]
    c1.plotly_chart(px.pie(sub, names="Subscription Status", values="Count",
                            title="Subscription Status", hole=0.4,
                            color_discrete_sequence=["#10b981", "#ef4444"]),
                    use_container_width=True)
    ss2 = eda_df.groupby("Subscription Status")[TARGET].mean().reset_index()
    ss2.columns = ["Subscription Status", "Avg (USD)"]
    ss2["Avg (USD)"] = ss2["Avg (USD)"].round(2)
    c2.plotly_chart(px.bar(ss2, x="Subscription Status", y="Avg (USD)",
                            title="Avg Spend by Subscription",
                            color="Subscription Status", text="Avg (USD)",
                            color_discrete_sequence=["#10b981", "#ef4444"]),
                    use_container_width=True)
    c3, c4 = st.columns(2)
    ds = eda_df.groupby("Discount Applied")[TARGET].mean().reset_index()
    ds.columns = ["Discount Applied", "Avg (USD)"]
    ds["Avg (USD)"] = ds["Avg (USD)"].round(2)
    c3.plotly_chart(px.bar(ds, x="Discount Applied", y="Avg (USD)",
                            title="Avg Spend: Discount Applied",
                            color="Discount Applied", text="Avg (USD)",
                            color_discrete_sequence=["#3b82f6", "#f59e0b"]),
                    use_container_width=True)
    ps2 = eda_df.groupby("Promo Code Used")[TARGET].mean().reset_index()
    ps2.columns = ["Promo Code Used", "Avg (USD)"]
    ps2["Avg (USD)"] = ps2["Avg (USD)"].round(2)
    c4.plotly_chart(px.bar(ps2, x="Promo Code Used", y="Avg (USD)",
                            title="Avg Spend: Promo Code",
                            color="Promo Code Used", text="Avg (USD)",
                            color_discrete_sequence=["#6366f1", "#ec4899"]),
                    use_container_width=True)
    st.plotly_chart(px.scatter(eda_df, x="Previous Purchases", y=TARGET,
                                title="Previous Purchases vs Purchase Amount",
                                opacity=0.5, trendline="ols",
                                color_discrete_sequence=["#8b5cf6"]),
                    use_container_width=True)


elif "Seasonal" in selected:
    st.title("Seasonal Trends")
    st.markdown("---")
    c1, c2 = st.columns(2)
    season_c = eda_df["Season"].value_counts().reset_index()
    season_c.columns = ["Season", "Count"]
    c1.plotly_chart(px.bar(season_c, x="Season", y="Count",
                            title="Purchases by Season", color="Season",
                            color_discrete_sequence=["#f59e0b","#10b981","#3b82f6","#ef4444"]),
                    use_container_width=True)
    season_s = eda_df.groupby("Season")[TARGET].mean().reset_index()
    season_s.columns = ["Season", "Avg (USD)"]
    season_s["Avg (USD)"] = season_s["Avg (USD)"].round(2)
    c2.plotly_chart(px.bar(season_s, x="Season", y="Avg (USD)",
                            title="Avg Spend by Season", color="Season", text="Avg (USD)",
                            color_discrete_sequence=["#f59e0b","#10b981","#3b82f6","#ef4444"]),
                    use_container_width=True)
    sc3 = eda_df.groupby(["Season", "Category"])[TARGET].count().reset_index()
    sc3.columns = ["Season", "Category", "Count"]
    st.plotly_chart(px.bar(sc3, x="Season", y="Count", color="Category",
                            title="Category Volume by Season", barmode="group"),
                    use_container_width=True)
    freq_c = eda_df["Frequency of Purchases"].value_counts().reset_index()
    freq_c.columns = ["Frequency", "Count"]
    st.plotly_chart(px.pie(freq_c, names="Frequency", values="Count",
                            title="Purchase Frequency", hole=0.35),
                    use_container_width=True)


elif "ML Model" in selected:
    st.title("ML Model Comparison")
    st.markdown("---")
    st.info("All metrics are computed on the held-out 20% test set.")
    st.dataframe(
        results_df.style.highlight_min(subset=["MAE", "RMSE"], color="#bbf7d0")
                        .highlight_max(subset=["R2"],          color="#bbf7d0"),
        use_container_width=True,
    )
    best2 = artifacts["best_model_name"]
    br = results_df[results_df["Model"] == best2].iloc[0]
    st.success(f"Best model: **{best2}**  |  MAE={br[\'MAE\']}  RMSE={br[\'RMSE\']}  R2={br[\'R2\']}")
    for metric, title in [("MAE", "MAE — lower is better"),
                           ("RMSE", "RMSE — lower is better"),
                           ("R2",   "R2 Score — higher is better")]:
        st.plotly_chart(px.bar(results_df, x="Model", y=metric,
                                title=title, color="Model", text=metric,
                                color_discrete_sequence=["#3b82f6","#10b981","#f59e0b"]),
                        use_container_width=True)


elif "Feature" in selected:
    st.title("Feature Importance")
    st.markdown("---")
    feat_names = artifacts["feature_names"]
    c1, c2 = st.columns(2)
    rf_imp = pd.DataFrame({"Feature": feat_names,
                            "Importance": artifacts["rf_model"].feature_importances_})\
               .sort_values("Importance", ascending=False).head(15)
    c1.plotly_chart(px.bar(rf_imp, x="Importance", y="Feature", orientation="h",
                            title="Random Forest Top 15",
                            color="Importance", color_continuous_scale="Blues"),
                    use_container_width=True)
    xgb_imp = pd.DataFrame({"Feature": feat_names,
                              "Importance": artifacts["xgb_model"].feature_importances_})\
                .sort_values("Importance", ascending=False).head(15)
    c2.plotly_chart(px.bar(xgb_imp, x="Importance", y="Feature", orientation="h",
                            title="XGBoost Top 15",
                            color="Importance", color_continuous_scale="Oranges"),
                    use_container_width=True)
    st.markdown("Higher importance = stronger influence on predicting Purchase Amount (USD).")


elif "Prediction" in selected:
    st.title("Live Purchase Amount Prediction")
    st.markdown("---")
    best_name_p   = artifacts["best_model_name"]
    best_model_p  = artifacts["best_model"]
    le_map        = artifacts["label_encoders"]
    scaler_p      = artifacts["scaler"]
    feat_names_p  = artifacts["feature_names"]
    cat_cols_p    = artifacts["categorical_cols"]
    num_cols_p    = artifacts["numerical_cols"]
    raw_p         = load_raw_data()

    st.info(f"Using best model: **{best_name_p}**")
    st.markdown("Fill in customer details and click **Predict**.")

    inputs = {}
    c1, c2, c3 = st.columns(3)
    with c1:
        if "Age" in feat_names_p:
            inputs["Age"] = st.slider("Age", int(raw_p["Age"].min()),
                                       int(raw_p["Age"].max()), int(raw_p["Age"].median()))
        if "Gender" in feat_names_p:
            inputs["Gender"] = st.selectbox("Gender", sorted(raw_p["Gender"].dropna().unique()))
        if "Item Purchased" in feat_names_p:
            inputs["Item Purchased"] = st.selectbox("Item Purchased",
                                                      sorted(raw_p["Item Purchased"].dropna().unique()))
        if "Category" in feat_names_p:
            inputs["Category"] = st.selectbox("Category",
                                               sorted(raw_p["Category"].dropna().unique()))
        if "Size" in feat_names_p:
            inputs["Size"] = st.selectbox("Size", sorted(raw_p["Size"].dropna().unique()))
    with c2:
        if "Color" in feat_names_p:
            inputs["Color"] = st.selectbox("Color", sorted(raw_p["Color"].dropna().unique()))
        if "Season" in feat_names_p:
            inputs["Season"] = st.selectbox("Season", sorted(raw_p["Season"].dropna().unique()))
        if "Location" in feat_names_p:
            inputs["Location"] = st.selectbox("Location",
                                               sorted(raw_p["Location"].dropna().unique()))
        if "Review Rating" in feat_names_p:
            inputs["Review Rating"] = st.slider("Review Rating", 1.0, 5.0,
                                                  float(raw_p["Review Rating"].median()), step=0.1)
        if "Previous Purchases" in feat_names_p:
            inputs["Previous Purchases"] = st.slider(
                "Previous Purchases",
                int(raw_p["Previous Purchases"].min()),
                int(raw_p["Previous Purchases"].max()),
                int(raw_p["Previous Purchases"].median()),
            )
    with c3:
        if "Subscription Status" in feat_names_p:
            inputs["Subscription Status"] = st.selectbox("Subscription Status", ["Yes", "No"])
        if "Payment Method" in feat_names_p:
            inputs["Payment Method"] = st.selectbox("Payment Method",
                                                      sorted(raw_p["Payment Method"].dropna().unique()))
        if "Shipping Type" in feat_names_p:
            inputs["Shipping Type"] = st.selectbox("Shipping Type",
                                                     sorted(raw_p["Shipping Type"].dropna().unique()))
        if "Discount Applied" in feat_names_p:
            inputs["Discount Applied"] = st.selectbox("Discount Applied", ["Yes", "No"])
        if "Promo Code Used" in feat_names_p:
            inputs["Promo Code Used"] = st.selectbox("Promo Code Used", ["Yes", "No"])
        if "Preferred Payment Method" in feat_names_p:
            inputs["Preferred Payment Method"] = st.selectbox(
                "Preferred Payment Method",
                sorted(raw_p["Preferred Payment Method"].dropna().unique())
            )
        if "Frequency of Purchases" in feat_names_p:
            inputs["Frequency of Purchases"] = st.selectbox(
                "Frequency of Purchases",
                sorted(raw_p["Frequency of Purchases"].dropna().unique())
            )

    st.markdown("---")
    if st.button("Predict Purchase Amount", type="primary", use_container_width=True):
        try:
            input_df = pd.DataFrame([{col: inputs.get(col, 0) for col in feat_names_p}])
            for col in cat_cols_p:
                if col in input_df.columns and col in le_map:
                    le = le_map[col]
                    val = str(input_df[col].iloc[0])
                    input_df[col] = le.transform([val]) if val in le.classes_ else [0]
            input_df[num_cols_p] = scaler_p.transform(input_df[num_cols_p])
            pred = max(0.0, float(best_model_p.predict(input_df)[0]))
            st.success(f"### Predicted Purchase Amount: **${pred:.2f} USD**")
            with st.expander("Input Summary"):
                st.dataframe(pd.DataFrame(list(inputs.items()),
                                          columns=["Feature", "Value"]),
                             use_container_width=True)
        except Exception as e:
            st.error(f"Prediction error: {e}")


st.markdown("---")
st.markdown(
    "<div style=\'text-align:center;color:#888;font-size:12px;\'>"
    "Customer Shopping Behavior & Purchase Amount Prediction &nbsp;|&nbsp; "
    "IBM SkillsBuild Data Analytics Internship &nbsp;|&nbsp; "
    "Built with Streamlit &amp; Python"
    "</div>",
    unsafe_allow_html=True,
)
'''

with open('app_combined.py', 'w', encoding='utf-8') as f:
    f.write(app_code.lstrip('\n'))

print('app_combined.py written successfully.')
print('To launch: streamlit run app_combined.py')

---
## Summary of Results

| Model | MAE | RMSE | R2 | Rank |
|---|---|---|---|---|
| Linear Regression | 20.8305 | **23.8560** | -0.0170 | ✅ Best |
| Random Forest | 20.8116 | 23.9098 | -0.0216 | 2nd |
| XGBoost | 22.3989 | 26.5306 | -0.2578 | 3rd |

> **Note:** All R² values are near-zero or negative because the dataset is synthetically generated with a uniformly distributed target variable ($20–$100). No regression model can find a meaningful signal when the target is uncorrelated with any feature. This is an honest, expected, and reportable finding — not a code error.

---

**Project:** Customer Shopping Behavior & Purchase Amount Prediction  
**Program:** IBM SkillsBuild Data Analytics Internship  
**Stack:** Python · Pandas · Scikit-learn · XGBoost · Streamlit · Plotly · Matplotlib